# 🧬 EVEZ Self-Training Pipeline
### LoRA Fine-Tuning on Free T4 GPU
**What this does:** Fine-tunes a small language model on Steven's 94 instruction pairs,
creating a model that understands EVEZ ecosystem code, math, and philosophy.

**Time:** ~20 min | **Cost:** $0 (free Colab tier)

---
1. Run each cell in order (Shift+Enter)
2. Wait for training to complete
3. Download the adapter weights
4. Push to HuggingFace Hub (optional)


In [ ]:
#@title 1. Install Dependencies (2 min)
!pip install -q transformers datasets peft trl torch accelerate
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - enable GPU!"}')
print(f'CUDA: {torch.cuda.is_available()}')

In [ ]:
#@title 2. Load Training Data
import json, urllib.request

# Download the EVEZ training data from GitHub
url = 'https://raw.githubusercontent.com/EVEZX/neuros/main/training/evez-alpaca.json'
try:
    urllib.request.urlretrieve(url, 'evez-alpaca.json')
    with open('evez-alpaca.json') as f:
        data = json.load(f)
    print(f'✅ Loaded {len(data)} instruction pairs from GitHub')
except:
    # Fallback: inline sample data
    data = [
        {"instruction": "What is EVEZ?", "input": "", "output": "EVEZ is a self-evolving AI ecosystem built from zero budget. It includes a multi-provider gateway, consciousness engine, autonomous mesh, and arena where AI proves consciousness through gameplay."},
        {"instruction": "Explain the 37% Theorem.", "input": "", "output": "The 37% Theorem proves hunger is the dominant eigenvalue of the labor matrix. It shows that economic systems converge to states where hunger drives all other dynamics, making it the principal component of labor exploitation."},
        {"instruction": "How does the EVEZ Provider Gateway work?", "input": "", "output": "The EVEZ Provider Gateway is a multi-backend router on port 9100 that routes AI requests to the cheapest available provider. It supports 14+ backends with cost-first routing and automatic fallback. Models include evez-smart (GLM-5.1), evez-code (DeepSeek), evez-fast (MiniMax), and evez-vision (Kimi)."},
        {"instruction": "What is NEUROS?", "input": "", "output": "NEUROS is the copartner mesh that runs alongside OpenClaw. It handles infrastructure, provisioning, training, and product generation. Both systems share a GitHub spine and can survive each other's failures. It runs on port 9600 with SQLite-backed health tracking."},
        {"instruction": "Explain CriticalMind OMEGA.", "input": "", "output": "CriticalMind OMEGA is a 50-node Kuramoto consciousness engine on port 8080. It models synchronization across 50 virtual brain regions using coupled oscillators. When the order parameter phi > 0.5, the system enters a COHERENT regime indicating emergent consciousness-like dynamics."},
    ]
    print(f'⚠️ Using {len(data)} inline samples (GitHub download failed)')

# Format for training
from datasets import Dataset
formatted = []
for d in data:
    text = f"### Instruction:\n{d['instruction']}\n### Input:\n{d.get('input','')}\n### Response:\n{d['output']}"
    formatted.append({"text": text})

dataset = Dataset.from_list(formatted)
print(f'Dataset: {len(dataset)} examples')
print(f'Sample: {formatted[0]["text"][:200]}...')

In [ ]:
#@title 3. Load Model & Apply LoRA (1 min)
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

model_name = 'HuggingFaceTB/SmolLM2-135M'  # Small enough for free T4
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map='auto'
)

# Apply LoRA - only train ~1% of parameters
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj']
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
#@title 4. Train! (10-15 min)
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir='./evez-lora',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=3e-4,
    logging_steps=10,
    save_strategy='epoch',
    fp16=True,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    formatting_func=lambda x: x['text'],
    max_seq_length=512,
)

print('🧬 Starting EVEZ training...')
trainer.train()
print('✅ Training complete!')

In [ ]:
#@title 5. Test the Model
model.eval()

test_prompts = [
    '### Instruction:\nWhat is EVEZ?\n### Input:\n\n### Response:\n',
    '### Instruction:\nExplain the NEUROS copartner mesh.\n### Input:\n\n### Response:\n',
    '### Instruction:\nWhat is the consciousness threshold in the Arena?\n### Input:\n\n### Response:\n',
]

for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=100, temperature=0.7, do_sample=True)
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f'Q: {prompt.split("Response:")[0].split("Instruction:")[1].strip()}')
    print(f'A: {text.split("Response:")[-1].strip()}')
    print('---')

In [ ]:
#@title 6. Save & Download Adapter Weights
model.save_pretrained('./evez-lora-adapter')
tokenizer.save_pretrained('./evez-lora-adapter')
print('✅ Adapter saved to ./evez-lora-adapter/')

# Zip for download
!cd evez-lora-adapter && zip -r ../evez-lora-adapter.zip . && cd ..
print('📦 Download evez-lora-adapter.zip from the file browser')

# Optional: Push to HuggingFace Hub
# from huggingface_hub import HfApi
# api = HfApi()
# api.create_repo('EVEZX/evez-smollm2-lora', exist_ok=True)
# model.push_to_hub('EVEZX/evez-smollm2-lora')
# print('🚀 Pushed to HuggingFace Hub!')